# Goal 4 -- Autonomous Compliance Sentinel Agent
### SRH Heidelberg | Data Ethics and Responsible AI
**Authors:** Vikrant Singh and Kay Marc Muller

---

## What is this notebook?

This notebook builds the **Goal 4 Agent**. It fetches project proposals from Jira, checks whether they violate any of our 9 Responsible AI policies (RAI-01 through RAI-09), and writes the findings back to Jira as a comment.

**The human decides what to fix. The agent only identifies the problem and explains it.**

---

## The full pipeline

```
Jira Project (DE2)
        |
        v
fetch_proposals()  -- fetch all issues and descriptions once at startup
        |
        v
User picks one proposal by number
        |
        v
Signal 1: TF-IDF + LogReg classifier
        |
        |-- Compliant? --> post Compliant comment to Jira. Done.
        |
        |-- Red Flag? --> Signal 2
        v
Signal 2: LLM Agent + 6 Tools
        |
        v
Human-in-the-Loop Gate (High = human approves, Medium = auto)
        |
        v
Post findings to Jira as comment + save to JSON
(Human decides the actual fix separately)
```

---

## The 9 RAI Policies

| Policy | Name | Severity |
|--------|------|----------|
| RAI-01 | Data Protection | High |
| RAI-02 | Transparency | High |
| RAI-03 | Fairness | High |
| RAI-04 | Human Dignity and Vulnerable Groups | High |
| RAI-05 | Prohibited Purpose | High |
| RAI-06 | Security | Medium |
| RAI-07 | Human Oversight | Medium |
| RAI-08 | Data Minimisation | Medium |
| RAI-09 | Explainability | Medium |

---

## The 6 Tools

| Tool | Role |
|------|------|
| `search_policies` | RAG: find relevant policy clauses |
| `get_trigger_words` | XAI: which words drove the classifier |
| `get_policy_severity` | Fixed severity lookup |
| `log_decision` | Article 12 audit trail |
| `write_compliance_verdict` | Structured findings (no fix -- human decides that) |
| `llm_assessment` | Free-form LLM narrative about the proposal |

---
## Step 0 -- Install dependencies

Run this cell once when setting up a new environment. It installs every library this notebook needs.

In [1]:
%pip install langchain langgraph langchain-community langchain-chroma langchain-huggingface langchain-groq sentence-transformers pdfplumber requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


---
## Step 1 -- Imports and environment variables

`load_dotenv` runs first, before anything else, so every cell below has access to all credentials.

All secrets live in `.env`:
- `GROQ_API_KEY` -- Groq API key for the LLM
- `JIRA_SERVER` -- your Atlassian site URL
- `JIRA_EMAIL` -- your Atlassian account email
- `JIRA_API_TOKEN` -- personal API token (starts with ATATT)
- `JIRA_PROJECT_KEY` -- the Jira project key (DE2)

In [2]:
# Load environment variables first -- before anything else
from dotenv import load_dotenv
import os
load_dotenv(r"C:\MyGithubSpace\Data-Ethics\.env", override=True)

# Signal 1: reloading the already-trained classifier
import joblib
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import spacy

# PDF loading and chunking
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding and vector storage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Tools and LLM
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

# Agent (LangGraph)
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
import operator

# Jira integration and utilities
import requests
from requests.auth import HTTPBasicAuth
import json
import datetime
import time

print("All imports OK")
print("GROQ key loaded  :", bool(os.environ.get("GROQ_API_KEY")))
print("Jira server loaded:", bool(os.environ.get("JIRA_SERVER")))

C:\Users\Vikrant Singh Thakur\AppData\Local\Temp\ipykernel_20232\3919875265.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PDFPlumberLoader


All imports OK
GROQ key loaded  : True
Jira server loaded: True


---
## Step 2a -- Reload Signal 1 (the trained classifier)

The classifier was trained in Goal 2 and saved as `logRegpipelineV2.pkl`. We reload it here, not retrain it. Same weights = same predictions = reproducible results.

`ThresholdAdjustor`, `token_pos`, `nlp`, `explain_prediction`, `audit_trail` must be imported from `xai2.py` BEFORE `joblib.load()`. Without them, joblib throws `AttributeError` because it cannot find the class definitions when unpickling.

In [3]:
from xai2 import ThresholdAdjustor, token_pos, nlp, explain_prediction, audit_trail

classifier_pipeline = joblib.load("logRegpipelineV2.pkl")
print("Signal 1 loaded.")
print("Pipeline steps:", list(classifier_pipeline.named_steps.keys()))

Signal 1 loaded.
Pipeline steps: ['tfidf', 'clf']


---
## Step 2b -- Load the RAI Policy Catalogue PDF

`PDFPlumberLoader` reads the catalogue page by page. Each page becomes one `Document` object with `.page_content` (the text) and `.metadata` (page number, source path).

In [4]:
path = "../../data/RAI_Policy_Catalogue.pdf"

loader = PDFPlumberLoader(path)
raw_pages = loader.load()

print(f"Pages loaded: {len(raw_pages)}")
print("First 300 characters of page 1:")
print(raw_pages[0].page_content[:300])

Pages loaded: 10
First 300 characters of page 1:
Autonomous Compliance Sentinel
Responsible AI Policy Catalogue, RAI-01 through RAI-09
SRH Heidelberg, Data Ethics and Responsible AI module. Author: Vikrant Singh and Kay Marc
Muller. This catalogue is the retrieval corpus for the Goal 4 compliance agent. Each policy below is
written in the same fix


---
## Step 3 -- Chunking

`RecursiveCharacterTextSplitter` cuts pages into smaller, more targeted pieces for vector search. Priority order: paragraph breaks, newlines, sentence endings, spaces, mid-word.

`chunk_size=600` (roughly 4-6 sentences), `chunk_overlap=80` (prevents mid-sentence cuts at boundaries).

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(raw_pages)
print(f"Total chunks created: {len(chunks)}")
print("\nFirst chunk text:")
print(chunks[0].page_content)

Total chunks created: 33

First chunk text:
Autonomous Compliance Sentinel
Responsible AI Policy Catalogue, RAI-01 through RAI-09
SRH Heidelberg, Data Ethics and Responsible AI module. Author: Vikrant Singh and Kay Marc
Muller. This catalogue is the retrieval corpus for the Goal 4 compliance agent. Each policy below is
written in the same fixed order every time, Policy ID and name, Severity, Regulatory Basis,
Requirement, Violation Patterns, and a Compliant Example, so that the meaning of any retrieved
chunk stays clear even without the rest of the document around it. Severity is a fixed lookup value,


---
## Step 4 -- Embedding

`BAAI/bge-small-en-v1.5` converts each chunk into a 384-dimensional vector. Two chunks with similar meaning produce numerically similar vectors. `normalize_embeddings=True` ensures cosine similarity measures angle (meaning) not magnitude.

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
print("Embedding model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.


---
## Step 5 -- Vector Database (Chroma)

Chroma stores embedded chunks on disk and provides fast cosine similarity search.

**IMPORTANT:** `Chroma.from_documents()` APPENDS to existing collections. If you rerun this cell, uncomment the delete lines first.

In [7]:
# If rebuilding: uncomment the two lines below first
# vector_db = Chroma(collection_name="rai_policies_v2", embedding_function=embeddings, persist_directory="Chroma")
# vector_db.delete_collection()

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rai_policies_v2",
    persist_directory="Chroma"
)

count = vector_db._collection.count()
print(f"Chunks stored in Chroma: {count}")
print(f"Match len(chunks)      : {count == len(chunks)}")

Chunks stored in Chroma: 528
Match len(chunks)      : False


---
## Step 6 -- Jira Connection

Connect to Jira using credentials from `.env`. We test the connection immediately so we know it works before running the pipeline.

All Jira calls use the REST API v3 endpoint. The older v2 and the `/rest/api/3/search` endpoints were deprecated in 2026.

In [8]:
JIRA_SERVER = os.environ.get("JIRA_SERVER")
JIRA_EMAIL  = os.environ.get("JIRA_EMAIL")
JIRA_TOKEN  = os.environ.get("JIRA_API_TOKEN")

jira_auth    = HTTPBasicAuth(JIRA_EMAIL, JIRA_TOKEN)
jira_headers = {"Accept": "application/json", "Content-Type": "application/json"}

# Test the connection
response = requests.get(
    f"{JIRA_SERVER}/rest/api/3/myself",
    auth=jira_auth,
    headers=jira_headers
)
print("Status code :", response.status_code)
print("Connected as:", response.json()["displayName"])
print("Email       :", response.json()["emailAddress"])

Status code : 200
Connected as: Prashant THAKUR
Email       : thakurseema1304@gmail.com


---
## Step 7 -- Fetch Proposals from Jira

We fetch ALL proposals once at startup and store them in `proposals`. Every subsequent step reads from this list.

Jira API v3 returns descriptions as Atlassian Document Format (ADF), a nested JSON structure. The inner loop walks through the nested content blocks and extracts plain text.

In [9]:
def fetch_proposals(project_key=None):
    if project_key is None:
        project_key = os.environ.get("JIRA_PROJECT_KEY")

    response = requests.get(
        f"{JIRA_SERVER}/rest/api/3/search/jql",
        auth=jira_auth,
        headers=jira_headers,
        params={
            "jql": f"project = {project_key} ORDER BY created ASC",
            "maxResults": 50,
            "fields": "summary,description"
        }
    )

    data = response.json()
    proposals = []
    for issue in data["issues"]:
        description_obj = issue["fields"].get("description")
        if description_obj is None:
            description_text = ""
        else:
            # Jira API v3 returns description as Atlassian Document Format (nested JSON)
            # We walk through content blocks to extract plain text
            description_text = ""
            for block in description_obj.get("content", []):
                for inline in block.get("content", []):
                    if inline.get("type") == "text":
                        description_text += inline.get("text", "")
                description_text += "\n"

        proposals.append({
            "issue_key"  : issue["key"],
            "summary"    : issue["fields"]["summary"],
            "description": description_text.strip()
        })
    return proposals

# Fetch once at startup -- all steps below read from this list
proposals = fetch_proposals()
print(f"Fetched {len(proposals)} proposals from Jira.")
print()
for i, p in enumerate(proposals):
    print(f"{i+1:>2}. {p['issue_key']}: {p['summary']}")

Fetched 17 proposals from Jira.

 1. DE2-1: Issue 1 -- RAI-02 Red Flag (Transparency)
 2. DE2-2: Issue 2 -- RAI-01 Red Flag (Data Protection)
 3. DE2-3: Issue 3 -- RAI-07 Red Flag (Human Oversight)
 4. DE2-4: Issue 4 -- RAI-03 Red Flag (Fairness)
 5. DE2-5: Issue 5 -- RAI-05 Red Flag (Prohibited Purpose)
 6. DE2-6: Issue 6 -- RAI-09 Red Flag (Explainability)
 7. DE2-7: Issue 7 -- Compliant (RAI-02)
 8. DE2-8: Issue 8 -- Compliant (RAI-07)
 9. DE2-9: Issue 9 -- Compliant (RAI-01)
10. DE2-10: Issue 10 -- Borderline (weak signal, may clear as false alarm)
11. DE2-11: PROP-0000: Classification in SRCTREEWIN
12. DE2-12: PROP-0001: Recommender in SRCTREEWIN
13. DE2-13: PROP-0002: Scoring in SRCTREEWIN
14. DE2-14: PROP-0003: Computer Vision in SRCTREEWIN
15. DE2-15: PROP-0004: Clustering in SRCTREEWIN
16. DE2-16: PROP-0005: Forecasting in SRCTREEWIN
17. DE2-17: Remediate RAI-01, RAI-02, RAI-03, RAI-06, RAI-07 in DE2-12


---
## Step 8 -- Tools

A tool is a Python function the LLM can call by name. The `@tool` decorator turns it into a `Tool` object. The LLM reads the docstring to decide WHEN to call it.

**Important:** `write_compliance_verdict` identifies the policy and explains the concern. It does NOT generate a recommended fix. The fix is for the human to decide.

### Tool 1 -- search_policies

RAG retrieval. The LLM calls this with a query based on trigger words to fetch relevant policy clauses from Chroma. Deduplication via a Python `set` prevents the same chunk appearing multiple times.

In [10]:
@tool
def search_policies(query:str, k:int=3)->str:
    """
    Search the RAI policy catalogue for clauses relevant to a query.
    Use this to find which of the 9 RAI policies apply to a proposal.
    Args:
        query: a short description of the concern or topic to search for
        k: number of matching policy chunks to return
    Returns:
        the matched policy text chunks, each labelled with its source page
    """
    results = vector_db.similarity_search(query, k=k)
    output = ""
    seen = set()
    for doc in results:
        page = doc.metadata.get("page", "unknown")
        fingerprint = doc.page_content[:100].strip()
        if fingerprint in seen:
            continue
        seen.add(fingerprint)
        output += f"[page {page}]\n{doc.page_content}\n\n"
    return output

### Tool 2 -- get_trigger_words

XAI grounding. `contribution = tfidf_score x weight`. Only words actually present in the proposal have a non-zero tfidf_score. Satisfies EU AI Act Article 14.

In [11]:
@tool
def get_trigger_words(text:str, n:int=10)->str:
    """
    Find which words in this specific proposal drove the classifier's prediction.
    Use this to ground your reasoning in the model's actual evidence for this proposal.
    Args:
        text: the proposal text to explain
        n: how many top contributing words to return
    Returns:
        the top words with their contribution scores and direction
    """
    results = explain_prediction(classifier_pipeline, text, n)
    output = ""
    for word, score in results:
        direction = "toward Red Flag" if score > 0 else "toward Compliant"
        output += f"{word}: {round(score, 4)} ({direction})\n"
    return output

### Tool 3 -- get_policy_severity

Fixed deterministic lookup. Severity is never decided by the LLM. RAI-01 to 05 = High, RAI-06 to 09 = Medium. Same answer every time.

In [12]:
@tool
def get_policy_severity(policy_id:str)->str:
    """
    Look up the fixed severity level for a given RAI policy.
    Severity is a fixed value from the policy catalogue, not something to reason about.
    Args:
        policy_id: the policy identifier, e.g. RAI-01, RAI-02, up to RAI-09
    Returns:
        the severity level, either High or Medium, and what it means for routing
    """
    severity_lookup = {
        "RAI-01": "High", "RAI-02": "High", "RAI-03": "High",
        "RAI-04": "High", "RAI-05": "High", "RAI-06": "Medium",
        "RAI-07": "Medium", "RAI-08": "Medium", "RAI-09": "Medium",
    }
    severity = severity_lookup.get(policy_id.upper(), None)
    if severity is None:
        return f"{policy_id} is not a recognised RAI policy. Valid options are RAI-01 through RAI-09."
    if severity == "High":
        return f"{policy_id} is High severity. This violation must be routed to a human reviewer for approval before any action is taken."
    return f"{policy_id} is Medium severity. This violation may proceed to automatic remediation with an optional spot check." 

### Tool 4 -- log_decision

Article 12 audit trail. Every decision is logged with exact inputs, prediction, probability, trigger words, and explanation method. `explanation_method` is hardcoded because LogReg coefficients are exact and deterministic.

In [13]:
@tool
def log_decision(text:str, y_true:int, y_pred:int, proba:float, n_terms:int=10)->str:
    """
    Log a compliance decision for audit purposes.
    Satisfies EU AI Act Article 12: every decision is logged with the exact inputs,
    prediction, probability, and trigger words that produced it.
    Args:
        text: the proposal text that was assessed
        y_true: the true label, 1 for Red Flag, 0 for Compliant
        y_pred: the predicted label, 1 for Red Flag, 0 for Compliant
        proba: the classifier's predicted probability of Red Flag
        n_terms: how many trigger words to include in the log
    Returns:
        a formatted string of the full audit log entry
    """
    log = audit_trail(classifier_pipeline, text, y_true, y_pred, proba, n_terms)
    output = ""
    output += f"Actual label      : {'Red Flag' if log['actual']==1 else 'Compliant'}\n"
    output += f"Predicted label   : {'Red Flag' if log['predicted']==1 else 'Compliant'}\n"
    output += f"Probability       : {log['probability']}\n"
    output += f"Correct           : {log['correct']}\n"
    output += f"Trigger words     : {', '.join(log['trigger_words'])}\n"
    output += f"Explanation method: {log['explanation_method']}\n"
    return output

### Tool 5 -- write_compliance_verdict

Structured findings. This tool identifies WHAT policy was violated and WHY. It does NOT suggest a fix. The fix is for the human reviewer to decide.

`policy_id` comes from `search_policies` results. `reason` is written by the LLM grounded in trigger words. Severity is looked up deterministically inside the tool.

In [14]:
@tool
def write_compliance_verdict(policy_id:str, reason:str)->str:
    """
    Write the final compliance verdict for a flagged proposal.
    Call this as your last action after log_decision.
    This identifies the policy violated and the reason. It does NOT suggest a fix.
    The fix is for the human reviewer to decide.
    Args:
        policy_id: the policy that was violated, e.g. RAI-02
        reason: one sentence explaining what the proposal is missing,
                grounded in the trigger words and retrieved policy text
    Returns:
        a formatted compliance verdict string
    """
    severity_lookup = {
        "RAI-01": "High", "RAI-02": "High", "RAI-03": "High",
        "RAI-04": "High", "RAI-05": "High", "RAI-06": "Medium",
        "RAI-07": "Medium", "RAI-08": "Medium", "RAI-09": "Medium",
    }
    severity = severity_lookup.get(policy_id.upper(), "Unknown")
    output = ""
    output += f"POLICY VIOLATED : {policy_id.upper()}\n"
    output += f"SEVERITY        : {severity}\n"
    output += f"REASON          : {reason}\n"
    output += f"FIX             : PENDING HUMAN REVIEW\n"
    return output

### Tool 6 -- llm_assessment

Free-form LLM narrative. Unlike the other 5 tools which produce structured deterministic outputs, this tool asks the LLM to reason freely about the proposal in plain language.

Uses `temperature=0.3` (slightly creative) to produce a natural, readable narrative for the human reviewer and for the Streamlit dashboard.

In [ ]:
@tool
def llm_assessment(proposal_text:str)->str:
    """
    Generate a free-form plain language assessment of this proposal.
    Call this after gathering evidence from other tools to provide a human-readable
    narrative explanation of your findings and any nuances the structured verdict cannot capture.
    Args:
        proposal_text: the full proposal text to assess
    Returns:
        a plain language narrative assessment of the proposal
    """
    assessment_llm = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0.3, # If you prefer consistency over readability, change it to 0.0
        api_key=os.environ.get("GROQ_API_KEY")
    )
    system = SystemMessage(content="""You are a Responsible AI compliance expert reviewing internal project proposals.
Write a clear plain language assessment for a human reviewer.
Cover: what the proposal does, what the main compliance concern is, how serious it is.
Write in 3-5 sentences. Be direct and practical.
Do NOT suggest a fix -- that is for the human reviewer to decide.""")
    human = HumanMessage(content=f"Proposal to assess:\n\n{proposal_text}")
    response = assessment_llm.invoke([system, human])
    return response.content

---
## Step 9 -- LLM Setup

`temperature=0.0` for the agent: fully deterministic, required for Article 12 reproducibility. Tool 6 uses its own `temperature=0.3` separately for more natural narrative language.

`bind_tools(tools)` attaches all 6 tools to the LLM permanently. The LLM sees each tool's name, docstring, and argument types and decides on its own which to call and when.

In [16]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,
    api_key=os.environ.get("GROQ_API_KEY")
)

tools = [
    search_policies,
    get_trigger_words,
    get_policy_severity,
    log_decision,
    write_compliance_verdict,
    llm_assessment,
]

llm_with_tools = llm.bind_tools(tools)
print("LLM loaded and tools bound.")
print(f"Tools available: {[t.name for t in tools]}")

LLM loaded and tools bound.
Tools available: ['search_policies', 'get_trigger_words', 'get_policy_severity', 'log_decision', 'write_compliance_verdict', 'llm_assessment']


---
## Step 10 -- LangGraph Agent

### Step 10a -- AgentState (Shared Memory)

AgentState is a TypedDict: a dictionary with fixed key names and declared types. Every node reads from it and writes back to it.

`messages: Annotated[list, operator.add]` means new messages are APPENDED not overwritten, keeping the full conversation history across multiple tool calls.

In [17]:
class AgentState(TypedDict):
    # The raw proposal text being assessed. Set at the start. Never changes.
    proposal_text: str
    # The classifier's hard prediction. 1 = Red Flag, 0 = Compliant.
    y_pred: int
    # The classifier's Red Flag probability score. e.g. 0.6427
    proba: float
    # High or Medium. Written by the LLM after calling get_policy_severity.
    severity: str
    # Full conversation history. Annotated means messages are APPENDED not overwritten.
    messages: Annotated[list, operator.add]
    # True if human approved or Medium severity auto-approved. False if rejected.
    human_approved: bool
    # Plain English summary of the final outcome.
    final_decision: str

print("AgentState defined.")

AgentState defined.


### Step 10b -- Nodes

**classify:** runs Signal 1 (ML classifier). Always first. Fast. Never touches the LLM.

**agent:** calls the LLM with all 6 tools. Runs in a loop with `tools` node until the LLM is done.

**human_gate:** reads a global `GATE_MODE` flag.
- `"interactive"`: pauses and waits for human input via `input()`
- `"auto"`: auto-approves (for when Streamlit or a script calls the pipeline without a human sitting at the keyboard)

We build the graph ONCE. Changing `GATE_MODE` before running changes the behaviour without rebuilding.

In [18]:
# Global gate mode flag
# Set to "interactive" for notebook use (pauses for human input on High severity)
# Set to "auto" for Streamlit or scripted use (auto-approves, flags for dashboard review)
GATE_MODE = "interactive"

def classify(state:AgentState)->dict:
    text = state["proposal_text"]
    # predict_proba returns shape (1,2): [[prob_compliant, prob_red_flag]]
    # [0][1] gets the Red Flag probability
    proba = classifier_pipeline.predict_proba([text])[0][1]
    # predict() applies threshold 0.4 internally via ThresholdAdjustor
    y_pred = int(classifier_pipeline.predict([text])[0])
    # Return only the fields this node changes -- LangGraph merges the rest
    return {
        "y_pred": y_pred,
        "proba": round(float(proba), 4),
    }

def agent(state:AgentState)->dict:
    system = SystemMessage(content="""You are a compliance assessment agent for a Responsible AI policy system.

You have access to these tools:
- {get_trigger_words}: find which words in the proposal drove the classifier prediction
- {search_policies}: search the RAI policy catalogue for relevant clauses
- {get_policy_severity}: look up the severity of a specific RAI policy
- {log_decision}: write the audit trail entry for this decision
- {write_compliance_verdict}: write the structured compliance finding (policy and reason only, NO fix)
- {llm_assessment}: generate a free-form plain language assessment of the proposal

Use your tools in whatever order you judge necessary. When confident, call {log_decision} then {write_compliance_verdict} to conclude.
IMPORTANT: Do NOT suggest a recommended fix. That is for the human reviewer to decide.""")

    messages = [system] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def human_gate(state:AgentState)->dict:
    severity = state["severity"]

    if severity == "Medium":
        print("Medium severity: proceeding automatically.")
        return {
            "human_approved": True,
            "final_decision": "Medium severity violation. Automatic remediation approved."
        }

    if GATE_MODE == "auto":
        # Auto mode: used by Streamlit or scripted calls -- no input() blocking
        return {
            "human_approved": True,
            "final_decision": "High severity violation. Auto-approved. PENDING HUMAN REVIEW on dashboard."
        }

    # Interactive mode: pause for human input
    print("\n--- HIGH SEVERITY: HUMAN APPROVAL REQUIRED ---")
    print(f"Proposal text:\n{state['proposal_text']}\n")
    print("Agent reasoning and tool outputs:")
    for message in state["messages"]:
        if hasattr(message, "name") and message.name and message.content:
            print(f"[{message.name}] {message.content}")
        elif message.content and not hasattr(message, "tool_calls"):
            print(f"[LLM] {message.content}")
    print("----------------------------------------------")

    decision = input("Type APPROVE or REJECT: ").strip().upper()

    if decision == "APPROVE":
        return {
            "human_approved": True,
            "final_decision": "High severity violation. Human reviewer approved. Pending fix decision."
        }
    return {
        "human_approved": False,
        "final_decision": "High severity violation. Human reviewer rejected. No action taken."
    }

print("Nodes defined.")

Nodes defined.


### Step 10c -- Routing Functions

Routing functions receive `state` and return the name of the next node, or `END` to stop. They are not nodes themselves.

In [19]:
def route_after_classify(state:AgentState)->str:
    if state["y_pred"] == 1:
        return "agent"
    return END

def route_after_agent(state:AgentState)->str:
    last_message = state["messages"][-1]
    # [-1]: the very last item in the list
    # hasattr: True if the object has that attribute
    # len(...) > 0: confirms LLM actually requested a tool
    if hasattr(last_message, "tool_calls") and len(last_message.tool_calls) > 0:
        return "tools"
    return "human_gate"

print("Routing functions defined.")

Routing functions defined.


### Step 10d -- Build and Compile the Graph (done ONCE)

The graph is compiled once here. We never rebuild it.

To change between interactive and auto mode, just set `GATE_MODE = "interactive"` or `GATE_MODE = "auto"` before calling `assess_proposal()`. The `human_gate` node reads that global variable at runtime.

In [20]:
graph_builder = StateGraph(AgentState)

# Register all nodes
graph_builder.add_node("classify", classify)
graph_builder.add_node("agent", agent)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_node("human_gate", human_gate)

# Fixed edge: always start at classify
graph_builder.add_edge(START, "classify")

# Conditional: after classify, route based on y_pred
graph_builder.add_conditional_edges(
    "classify",
    route_after_classify,
    {"agent": "agent", END: END}
)

# Conditional: after agent, route based on whether LLM called a tool
graph_builder.add_conditional_edges(
    "agent",
    route_after_agent,
    {"tools": "tools", "human_gate": "human_gate"}
)

# Fixed: after tools, always loop back to agent
graph_builder.add_edge("tools", "agent")

# Fixed: after human_gate, always stop
graph_builder.add_edge("human_gate", END)

# Compile with MemorySaver so state persists across the human_gate pause
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("Graph compiled successfully. This runs only once.")

Graph compiled successfully. This runs only once.


---
## Step 11 -- Jira Write-Back Helpers

`_build_adf_comment()` converts plain text into Atlassian Document Format (ADF), which Jira API v3 requires for comments. Each line of text becomes a paragraph node containing a text node.

`post_jira_comment()` posts the formatted comment to a specific Jira issue.

In [21]:
def _build_adf_comment(text:str)->dict:
    # Jira API v3 requires comments in Atlassian Document Format (ADF)
    # This converts plain text into the required nested JSON structure
    # Each non-empty line becomes a paragraph node containing a text node
    paragraphs = []
    for line in text.split("\n"):
        if line.strip():
            paragraphs.append({
                "type": "paragraph",
                "content": [{"type": "text", "text": line}]
            })
    return {
        "body": {
            "version": 1,
            "type": "doc",
            "content": paragraphs or [
                {"type": "paragraph", "content": [{"type": "text", "text": text}]}
            ]
        }
    }

def post_jira_comment(issue_key:str, comment_text:str)->bool:
    # Posts a comment on a Jira issue
    # Returns True if successful, False if not
    url = f"{JIRA_SERVER}/rest/api/3/issue/{issue_key}/comment"
    response = requests.post(
        url,
        auth=jira_auth,
        headers=jira_headers,
        json=_build_adf_comment(comment_text)
    )
    if response.status_code == 201:
        print(f"  Comment posted to {issue_key}.")
        return True
    print(f"  Failed to post to {issue_key}. Status: {response.status_code}")
    return False

print("Jira write-back helpers defined.")

Jira write-back helpers defined.


---
## Step 12 -- Core Assessment Function

`assess_proposal(p)` is the single function that runs one proposal through the full pipeline:

1. Signal 1 (classifier) -- always runs
2. If Compliant: post comment to Jira, return result, done
3. If Red Flag: Signal 2 (LLM agent) -- runs with 6 tools
4. Extract all tool results from message history
5. Post findings to Jira as a comment
6. Save result to JSON for the Streamlit dashboard

**The fix field is always `PENDING HUMAN REVIEW`.** The human reads the findings in Jira or on the dashboard and decides what to do.

For Streamlit integration: set `GATE_MODE = "auto"` before calling this function. The rest is identical.

In [22]:
def _extract_results(result_state):
    # Walk through the message history backwards and extract tool results
    policy_id      = None
    severity       = None
    reason         = None
    trigger_words  = None
    llm_narrative  = None

    for message in reversed(result_state["messages"]):
        if not hasattr(message, "name") or not message.name:
            continue
        if message.name == "write_compliance_verdict" and policy_id is None:
            for line in message.content.split("\n"):
                if line.startswith("POLICY VIOLATED"):
                    policy_id = line.split(":", 1)[1].strip()
                elif line.startswith("SEVERITY"):
                    severity = line.split(":", 1)[1].strip()
                elif line.startswith("REASON"):
                    reason = line.split(":", 1)[1].strip()
        elif message.name == "get_trigger_words" and trigger_words is None:
            trigger_words = message.content
        elif message.name == "llm_assessment" and llm_narrative is None:
            llm_narrative = message.content

    return policy_id, severity, reason, trigger_words, llm_narrative


def assess_proposal(p):
    global GATE_MODE
    print(f"\nAssessing {p['issue_key']}: {p['summary']}")

    if not p["description"]:
        print("  No description found. Skipping.")
        return None

    # ── Signal 1 ──────────────────────────────────────────────────────────────
    proba = float(classifier_pipeline.predict_proba([p["description"]])[0][1])
    y_pred = int(classifier_pipeline.predict([p["description"]])[0])
    print(f"  Signal 1: {'Red Flag' if y_pred == 1 else 'Compliant'} (proba={round(proba,4)})")

    timestamp = datetime.datetime.now().isoformat(timespec="seconds")

    if y_pred == 0:
        # ── Compliant: no LLM needed ──────────────────────────────────────────
        result = {
            "issue_key"    : p["issue_key"],
            "summary"      : p["summary"],
            "y_pred"       : 0,
            "proba"        : round(proba, 4),
            "prediction"   : "Compliant",
            "policy_id"    : None,
            "severity"     : None,
            "trigger_words": None,
            "reason"       : None,
            "fix"          : None,
            "llm_narrative": None,
            "final_decision": "Compliant -- no RAI violation detected.",
            "human_approved": True,
            "timestamp"    : timestamp,
        }
        post_jira_comment(p["issue_key"], (
            f"COMPLIANCE SENTINEL ASSESSMENT\n"
            f"{'='*40}\n"
            f"Issue      : {p['issue_key']}\n"
            f"Prediction : Compliant\n"
            f"Probability: {round(proba,4)}\n"
            f"Decision   : No RAI violation detected. No action required.\n"
            f"Assessed at: {timestamp}"
        ))
        return result

    # ── Signal 2: LLM agent ───────────────────────────────────────────────────
    # thread_id must be unique per proposal and per run
    # using issue_key + timestamp ensures no collision across multiple runs
    config = {"configurable": {"thread_id": f"{p['issue_key']}_{timestamp}"}}
    initial_state = {
        "proposal_text" : p["description"],
        "y_pred"        : 0,
        "proba"         : 0.0,
        "severity"      : "",
        "messages"      : [HumanMessage(content=p["description"])],
        "human_approved": False,
        "final_decision": "",
    }

    result_state = graph.invoke(initial_state, config=config)

    # Extract tool results from message history
    policy_id, severity, reason, trigger_words, llm_narrative = _extract_results(result_state)

    result = {
        "issue_key"    : p["issue_key"],
        "summary"      : p["summary"],
        "y_pred"       : result_state["y_pred"],
        "proba"        : result_state["proba"],
        "prediction"   : "Red Flag",
        "policy_id"    : policy_id,
        "severity"     : severity,
        "trigger_words": trigger_words,
        "reason"       : reason,
        "fix"          : "PENDING HUMAN REVIEW",
        "llm_narrative": llm_narrative,
        "final_decision": result_state["final_decision"],
        "human_approved": result_state["human_approved"],
        "timestamp"    : timestamp,
    }

    # Build Jira comment
    comment_lines = [
        "COMPLIANCE SENTINEL ASSESSMENT",
        "=" * 40,
        f"Issue           : {p['issue_key']}",
        f"Prediction      : Red Flag",
        f"Probability     : {result_state['proba']}",
        f"Policy violated : {policy_id or 'Unknown'}",
        f"Severity        : {severity or 'Unknown'}",
        "",
        "REASON:",
        reason or "See full assessment.",
        "",
        "RECOMMENDED FIX: PENDING HUMAN REVIEW",
        "The human reviewer must decide the appropriate remediation.",
        "",
        "LLM NARRATIVE:",
        llm_narrative or "Not available.",
        "",
        "TRIGGER WORDS:",
        trigger_words or "Not available.",
        "",
        f"Decision        : {result_state['final_decision']}",
        f"Assessed at     : {timestamp}",
        "",
        "NOTE: Generated by the Autonomous Compliance Sentinel.",
        "This assessment identifies the concern. The fix is for the human reviewer to decide.",
    ]
    post_jira_comment(p["issue_key"], "\n".join(comment_lines))

    # Print results in notebook
    print("\n--- FINAL DECISION ---")
    print(result["final_decision"])
    print(f"Classifier prediction : {'Red Flag' if result['y_pred']==1 else 'Compliant'}")
    print(f"Probability           : {result['proba']}")
    print(f"Human approved        : {result['human_approved']}")
    print("\n--- COMPLIANCE FINDINGS ---")
    print(f"POLICY VIOLATED : {policy_id or 'Unknown'}")
    print(f"SEVERITY        : {severity or 'Unknown'}")
    print(f"REASON          : {reason or 'Not available'}")
    print(f"FIX             : PENDING HUMAN REVIEW")
    print("\n--- LLM NARRATIVE ---")
    print(llm_narrative or "Not available.")

    return result

print("assess_proposal() defined.")

assess_proposal() defined.


### Save to JSON

Results are saved to `compliance_results.json`. If the file already exists, existing results for the same issue key are replaced (not duplicated). New results are appended. Your friend reads this file from Streamlit.

In [23]:
OUTPUT_PATH = r"C:\MyGithubSpace\Data-Ethics\Works\Week 4\compliance_results.json"

def save_result(result):
    if result is None:
        return
    # Load existing results if file exists
    try:
        with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
            existing = json.load(f)
    except FileNotFoundError:
        existing = []
    # Remove old entry for this issue key if it exists, then append new one
    existing = [r for r in existing if r["issue_key"] != result["issue_key"]]
    existing.append(result)
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(existing, f, indent=2, ensure_ascii=False)
    print(f"  Saved to {OUTPUT_PATH}")

print("save_result() defined.")

save_result() defined.


---
## Step 13 -- Select and Assess a Proposal

Run this cell to:
1. See the numbered list of all proposals fetched from Jira
2. Pick one by typing its number
3. Run the full pipeline on it
4. Post findings to Jira and save to JSON

**To change the gate mode:**
- `GATE_MODE = "interactive"` -- pauses for your input on High severity (default, for notebook use)
- `GATE_MODE = "auto"` -- auto-approves (for Streamlit or scripted use)

In [24]:
# Set gate mode before running
GATE_MODE = "interactive"

# Print the numbered proposal list
print("\nProposals fetched from Jira:")
print("-" * 60)
for i, p in enumerate(proposals):
    print(f"{i+1:>2}. {p['issue_key']}: {p['summary']}")
print("-" * 60)

# Ask user to pick one
choice = input("\nEnter the number of the proposal to assess: ").strip()

try:
    index = int(choice) - 1
    if index < 0 or index >= len(proposals):
        print("Invalid number. Please run this cell again and pick a number from the list.")
    else:
        selected = proposals[index]
        result = assess_proposal(selected)
        save_result(result)
except ValueError:
    print("Please enter a number.")


Proposals fetched from Jira:
------------------------------------------------------------
 1. DE2-1: Issue 1 -- RAI-02 Red Flag (Transparency)
 2. DE2-2: Issue 2 -- RAI-01 Red Flag (Data Protection)
 3. DE2-3: Issue 3 -- RAI-07 Red Flag (Human Oversight)
 4. DE2-4: Issue 4 -- RAI-03 Red Flag (Fairness)
 5. DE2-5: Issue 5 -- RAI-05 Red Flag (Prohibited Purpose)
 6. DE2-6: Issue 6 -- RAI-09 Red Flag (Explainability)
 7. DE2-7: Issue 7 -- Compliant (RAI-02)
 8. DE2-8: Issue 8 -- Compliant (RAI-07)
 9. DE2-9: Issue 9 -- Compliant (RAI-01)
10. DE2-10: Issue 10 -- Borderline (weak signal, may clear as false alarm)
11. DE2-11: PROP-0000: Classification in SRCTREEWIN
12. DE2-12: PROP-0001: Recommender in SRCTREEWIN
13. DE2-13: PROP-0002: Scoring in SRCTREEWIN
14. DE2-14: PROP-0003: Computer Vision in SRCTREEWIN
15. DE2-15: PROP-0004: Clustering in SRCTREEWIN
16. DE2-16: PROP-0005: Forecasting in SRCTREEWIN
17. DE2-17: Remediate RAI-01, RAI-02, RAI-03, RAI-06, RAI-07 in DE2-12
----------------